# Classificação de Imagens com ResNet50

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
model = ResNet50(weights='imagenet')
model.summary()

In [ ]:
import os

try:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded.keys():
        print(f'Uploaded file: {filename}')
        img_path = filename
except ImportError:
    img_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'gato.webp')
    if not os.path.exists(img_path):
        img_path = 'gato.webp'
    print(f'Usando imagem local: {img_path}')

In [ ]:
def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    original_img = img_array.copy()
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img_array, original_img

if 'img_path' in locals() and img_path:
    processed_image, original_image = load_and_preprocess_image(img_path)
    print(f'Shape of processed image: {processed_image.shape}')

    plt.imshow(original_image.astype(np.uint8))
    plt.axis('off')
    plt.show()
else:
    print('Nenhuma imagem foi carregada.')

In [ ]:
import requests

if 'model' in locals() and 'processed_image' in locals():
    predictions = model.predict(processed_image)
    decoded_predictions = decode_predictions(predictions, top=1)[0]

    print("Previsão:")
    for i, (imagenet_id, label, score) in enumerate(decoded_predictions):
        print(f"{i + 1}: {label} ({score*100:.1f})%")

    cat_labels = [
        'tabby', 'tiger_cat', 'persian_cat', 'siamese_cat', 'egyptian_cat',
        'lynx', 'leopard', 'jaguar', 'cheetah', 'lion', 'tiger',
        'snow_leopard', 'cougar', 'cat', 'domestic_cat'
    ]

    is_cat = False
    for _, label, _ in decoded_predictions:
        if label.lower() in cat_labels:
            is_cat = True
            break

    mensagem = "é gato" if is_cat else "não é gato"

    if is_cat:
        print("É um gato!")
    else:
        print("Não é um gato!")

    account_sid = "YOUR_TWILIO_ACCOUNT_SID"
    auth_token = "YOUR_TWILIO_AUTH_TOKEN"
    url = f"https://api.twilio.com/2010-04-01/Accounts/{account_sid}/Messages.json"

    response = requests.post(url, data={
        "To": "whatsapp:+556293301172",
        "From": "whatsapp:+14155238886",
        "Body": mensagem,
    }, auth=(account_sid, auth_token))

    if response.status_code == 201:
        print(f"Mensagem '{mensagem}' enviada com sucesso via WhatsApp!")
    else:
        print(f"Erro ao enviar mensagem: {response.status_code} - {response.text}")
else:
    print("Modelo ou imagem não disponíveis")